In [2]:
import requests
import pandas as pd
import json
import time
from config import *

The purpose of this notebook was to test the retrieval of the teachers using a few API calls before automatically performing multiple calls at scale

API URL

In [3]:
italki = "https://api.italki.com/api/v2/teachers"

Define the next few functions

In [4]:
def get_data(page, min_price, max_price):
    headers_format = {
        'Accept': 'application/json, text/plain, */*',
        'Accept-Encoding': 'gzip, deflate, br, zstd',
        'Accept-Language': 'en-GB,en-US;q=0.9,en;q=0.8,zh-CN;q=0.7,zh;q=0.6',
        'Content-Length': '130',
        'Content-Type': 'application/json',
        'Origin': 'https://www.italki.com',
        'Priority': 'u=1, i',
        'Referer': 'https://www.italki.com/',
        'Sec-Ch-Ua': '"Chromium";v="148", "Google Chrome";v="148", "Not/A)Brand";v="99"',
        'Sec-Ch-Ua-Platform': '?0',
        'Sec-Ch-Ua-Platform': "Windows",
        'Sec-Fetch-Dest': '',
        'Sec-Fetch-Mode': 'cors',
        'Sec-Fetch-Site': 'same-site',
        'traceparent': traceparent,
        'User-Agent': personal_user_agent,
        'x-browser-key': personal_browser_key, # Found in the actual POST request
        'x-device': '10',                       # Common default, but check yours
        'x-locale': 'en',
        'x-token': personal_x_token    
    }

    payload = {
        "teach_language": {
            "language": "french",
            "max_price": max_price,
            "min_price": min_price,
        },
        "page": page,
        "page_size": 20,
        "user_timezone": "Asia/Singapore"
    }

    response = requests.post(url = italki, headers = headers_format, json=payload)
    time.sleep(2)
    dictionary = response.json()
    return dictionary

In [5]:
def join_data(new_data, filename):
    with open(filename, 'r') as file:
        existing = json.load(file)

    existing_tuples = {tuple(sorted(d.items())) for d in existing}
    new_no_duplicates = []
    seen = set()
    for d in new_data:
        d_tuple = tuple(sorted(d.items()))

        if d_tuple not in seen:
            new_no_duplicates.append(d)
            seen.add(d_tuple)

    new_to_join = [
        d for d in new_no_duplicates
        if tuple(sorted(d.items())) not in existing_tuples
    ]    
    
    combined = existing + new_to_join
    with open(filename, 'w') as file:
        json.dump(combined, file, indent=4)

In [6]:
def override_data(new_data, filename):
    with open(filename, 'w') as file:
        json.dump(new_data, file, indent=4)

Check number of entries per price range

In [8]:
# Change price here
min_price = 500
max_price = min_price + 99
def check_entries(min_price, max_price):
    dictionary = get_data(1, min_price, max_price)
    paging = dictionary['paging']
    return paging

page = check_entries(min_price, max_price)
print(page)
if int(page['total']) % 20 == 0:
    range_until = (int(page['total']) // 20) + 1
else:
    range_until = (int(page['total']) // 20) + 2
print(range_until)

{'page': 1, 'page_size': 20, 'total': 11, 'has_next': 0}
2


Get price range

In [9]:
print(min_price, max_price, range_until)

500 599 2


Appending new information to old JSONs

In [10]:
def user_course_info(teachers):
    user_course_info_list = []
    for teacher in teachers:
        user_course_info = teacher['user_info'] | teacher['course_info']
        del user_course_info['avatar_file_name']
        del user_course_info['is_online']
        del user_course_info['last_login_time']
        del user_course_info['trial_description']
        user_course_info_list.append(user_course_info)
    return user_course_info_list

    # user_course_info
    # df = pd.DataFrame(user_course_info_list)


In [11]:
def teacher_stats(teachers):
    teacher_stats_list = []
    for teacher in teachers:
        test_stat = {}
        test_stat['user_id'] = teacher['user_info']['user_id']
        test_stat['finished_session'] = teacher['teacher_statistics']['finished_session']
        test_stat['response_rate'] = teacher['teacher_statistics']['response_rate']
        test_stat['attendance_rate'] = teacher['teacher_statistics']['attendance_rate']
        teacher_stats_list.append(test_stat)
    return teacher_stats_list

In [12]:
def pro_course_prices(teachers):
    master_price_list = []
    pro_course_detail_list = []
    for teacher in teachers:
        for course in teacher['pro_course_detail']:
            for prices in course['price_list']:
                master_price_list.append(prices)
            course_no_price = course.copy()
            del course_no_price['price_list']
            del course_no_price['description']
            del course_no_price['level_lower_limit']
            del course_no_price['level_up_limit']
            del course_no_price['course_category']
            del course_no_price['course_tags']
            del course_no_price['create_time']
            pro_course_detail_list.append(course_no_price)
    return master_price_list, pro_course_detail_list


In [13]:
with open('also_speaks_reference.json', 'r') as file:
    also_speak_index_list = json.load(file)
def check_language(language):
    if language in also_speak_index_list:
        return also_speak_index_list.index(language)
    else:
        also_speak_index_list.append(language)
        return also_speak_index_list.index(language)

In [14]:
def also_speak(teachers):
    also_speak_list = []
    for teacher in teachers:
        user_id = teacher['user_info']['user_id']
        for language in teacher['teacher_info']['also_speak']:
            current_language = language['language']
            index = check_language(current_language)
            id_language = {}
            id_language['user_id'] = user_id
            id_language['language'] = index
            # id_language['map'] = index
            also_speak_list.append(id_language)
    return also_speak_list

Loop through pages

In [ ]:
# english_list = []
# for price in range(450, 550, 50):
#     has_next = 1
#     page = 1
#     while has_next == 1:
#         item = get_data(page, price, price + 49)
#         english_list.append(item)
#         page += 1
#         has_next = item['paging']['has_next']
# # json_new(1, dictionary)
# english_list

In [ ]:
user_course_info_cycle = []
teacher_stats_cycle = []
pro_course_cycle = []
price_list_cycle = []
also_speak_list_cycle = []

has_next = 1
page = 1
print(min_price, max_price, range_until)
while has_next == 1:
    api_call = get_data(page, min_price, max_price)
    teachers = api_call['data']
    user_course_info_cycle += user_course_info(teachers)
    teacher_stats_cycle += teacher_stats(teachers)
    price_list_buffer, pro_course_buffer = pro_course_prices(teachers)
    pro_course_cycle += pro_course_buffer
    price_list_cycle += price_list_buffer
    also_speak_list_cycle += also_speak(teachers)

    
    page += 1
    has_next = api_call['paging']['has_next']
    print('page', page)

join_data(user_course_info_cycle, 'user_course_info.json')
join_data(teacher_stats_cycle, 'teacher_stats.json')
join_data(pro_course_cycle, 'pro_course.json')
join_data(price_list_cycle, 'price_list.json')
join_data(also_speak_list_cycle, 'also_speaks.json')
override_data(also_speak_index_list, 'also_speaks_reference.json')
# df = pd.DataFrame(user_course_info_list)


500 599 2
page 2


In [38]:
with open('also_speaks.json', 'r') as file:
    also_speak_list = json.load(file)

with open('also_speaks_reference.json', 'r') as file:
    also_speaks_index_list = json.load(file)

df_also_speak_list = pd.DataFrame(also_speak_list)
df_languages = pd.DataFrame(also_speak_index_list, columns=['other languages'])
result = pd.merge(df_also_speak_list, df_languages, left_on='language', right_index=True, how="left")
result

,user_id,language,other languages
0,5467830,0,spanish
1,5467830,1,arabic
2,5467830,2,arabic(maghrebi)
3,11401921,3,filipino(tagalog)
4,11401921,4,japanese
...,...,...,...
146,30752880,38,greek
147,30752880,1,arabic
148,30752880,12,german
149,30752880,4,japanese


In [16]:
with open('user_course_info.json', 'r') as file:
    user_course_info_check = json.load(file)

df_user_course_info_check = pd.DataFrame(user_course_info_check)

In [56]:
df_user_course_info_check[df_user_course_info_check['user_id'].duplicated(keep=False)]

,user_id,nickname,is_tutor,is_pro,origin_country_id,living_country_id,origin_city_id,origin_city_name,living_city_id,living_city_name,timezone,trial_length,has_trial,trial_price,min_price,trial_session_count,has_beginner_course


In [17]:
df_user_course_info_check

,user_id,nickname,is_tutor,is_pro,origin_country_id,living_country_id,origin_city_id,origin_city_name,living_city_id,living_city_name,timezone,trial_length,has_trial,trial_price,min_price,trial_session_count,has_beginner_course
0,5467830,⭐Learn with Kim/Kin,1,0,CN,KR,CN00001,Shanghai,KR00001,Seoul,Asia/Shanghai,2,0,1988,500,562,1
1,11401921,Teacher Emee,1,0,PH,PH,PH00000,Other,PH00000,Other,Asia/Manila,2,0,500,500,186,1
2,9627036,Shyam Syangtan,1,0,IN,IN,IN00771,Sonipat,IN00231,Delhi,Asia/Kolkata,2,0,700,500,1366,1
3,5787377,Kimi(y)a 👩🏻🎓❤️,1,0,IR,IT,IR00016,Tabriz,IT00016,Turin,Asia/Tehran,2,0,799,500,384,1
4,9934037,Anne Rola,1,0,PH,PH,PH00000,Other,PH00236,Tacloban,Asia/Shanghai,2,0,500,500,115,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
389,25268520,Nestor⭐⭐⭐⭐⭐,1,0,CM,CM,CM00000,Other,CM00006,Yaounde,Africa/Douala,3,0,500,500,30,1
390,25732646,LINCHA DENISE MADO,1,0,CM,CM,CM00000,Other,CM00006,Yaounde,Africa/Douala,2,0,500,500,5,1
391,31616769,Casseus Fodley,1,0,HT,HT,HT00001,Port-au-Prince,HT00021,Petionville,America/Port-au-Prince,2,0,500,500,0,0
392,25732679,Syntia,1,0,CM,CM,CM00006,Yaounde,CM00006,Yaounde,Africa/Douala,2,0,500,500,21,1
